In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor
from prophet import Prophet

from tqdm import tqdm

# ============================================================
# LOAD PROCESSED DATA
# ============================================================

base_data = pd.read_csv("processed_base_dataset.csv", parse_dates=["Date"])
alt_data  = pd.read_csv("processed_alt_dataset.csv", parse_dates=["Date"])

HORIZON = 10
QUANTILES = [0.1, 0.5, 0.9]

# ============================================================
# METRIC FUNCTIONS
# ============================================================

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def pinball_loss(y_true, y_pred, q):
    """
    Computes pinball loss for a single quantile.
    """
    error = y_true - y_pred
    return np.mean(np.maximum(q * error, (q - 1) * error))


def crps_approximation(y_true, q_preds, quantiles):
    """
    Approximate CRPS by averaging pinball losses across quantiles.
    """
    total = 0
    for q, pred in zip(quantiles, q_preds):
        total += pinball_loss(y_true, pred, q)
    return total / len(quantiles)


def picp(y_true, lower, upper):
    """
    Prediction Interval Coverage Probability.
    """
    inside = np.logical_and(y_true >= lower, y_true <= upper)
    return np.mean(inside)


def interval_width(lower, upper):
    return np.mean(upper - lower)

# ============================================================
# WALK-FORWARD VALIDATION FUNCTION
# ============================================================

def walk_forward(dataset, model_type="qrf"):
    
    results = []
    
    split_index = int(len(dataset) * 0.8)
    
    for h in tqdm(range(1, HORIZON + 1), desc="Horizons"):
        
        target_col = f"Target_t+{h}"
        
        X = dataset.drop(columns=["Date"] + 
                         [f"Target_t+{i}" for i in range(1, HORIZON+1)])
        y = dataset[target_col]
        
        X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
        y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]
        
        quantile_predictions = []
        
        for q in QUANTILES:
            
            if model_type == "qrf":
                
                model = RandomForestRegressor(
                    n_estimators=300,
                    min_samples_leaf=5,
                    random_state=42
                )
                
                model.fit(X_train.values, y_train.values)

                # Collect predictions from each tree
                all_tree_preds = np.array([
                    tree.predict(X_test.values)
                    for tree in model.estimators_
                ])

                # Compute quantile per observation
                preds = np.percentile(
                    all_tree_preds,
                    q * 100,
                    axis=0
                )
                
                preds = np.mean(preds)
                preds = np.repeat(preds, len(y_test))
                
            elif model_type == "xgb":
                
                model = XGBRegressor(
                    objective="reg:quantileerror",
                    quantile_alpha=q,
                    n_estimators=300,
                    learning_rate=0.05,
                    max_depth=4,
                    random_state=42
                )
                
                model.fit(X_train.values, y_train.values)
                preds = model.predict(X_test.values)
                
            quantile_predictions.append(preds)
        
        lower = quantile_predictions[0]
        median = quantile_predictions[1]
        upper = quantile_predictions[2]
        
        crps_val = crps_approximation(y_test, quantile_predictions, QUANTILES)
        picp_val = picp(y_test, lower, upper)
        width_val = interval_width(lower, upper)
        rmse_val = rmse(y_test, median)
        
        results.append({
            "Horizon": h,
            "CRPS": crps_val,
            "PICP": picp_val,
            "Interval_Width": width_val,
            "RMSE": rmse_val
        })
    
    return pd.DataFrame(results)

# ============================================================
# RUN MODELS
# ============================================================

print("Running QRF (Base Dataset)...")
qrf_base_results = walk_forward(base_data, model_type="qrf")

print("Running QRF (Alternative Dataset)...")
qrf_alt_results = walk_forward(alt_data, model_type="qrf")

print("Running XGBoost (Base Dataset)...")
xgb_base_results = walk_forward(base_data, model_type="xgb")

print("Running XGBoost (Alternative Dataset)...")
xgb_alt_results = walk_forward(alt_data, model_type="xgb")

# ============================================================
# DISPLAY RESULTS TABLES
# ============================================================

print("\nQRF - Base Dataset")
display(qrf_base_results)

print("\nQRF - Alternative Dataset")
display(qrf_alt_results)

print("\nXGB - Base Dataset")
display(xgb_base_results)

print("\nXGB - Alternative Dataset")
display(xgb_alt_results)

# ============================================================
# PLOT RMSE COMPARISON
# ============================================================

plt.figure()
plt.plot(qrf_base_results["Horizon"], qrf_base_results["RMSE"])
plt.plot(qrf_alt_results["Horizon"], qrf_alt_results["RMSE"])
plt.title("QRF RMSE Comparison")
plt.xlabel("Forecast Horizon")
plt.ylabel("RMSE")
plt.legend(["Base", "Alternative"])
plt.show()

plt.figure()
plt.plot(xgb_base_results["Horizon"], xgb_base_results["RMSE"])
plt.plot(xgb_alt_results["Horizon"], xgb_alt_results["RMSE"])
plt.title("XGBoost RMSE Comparison")
plt.xlabel("Forecast Horizon")
plt.ylabel("RMSE")
plt.legend(["Base", "Alternative"])
plt.show()


Running QRF (Base Dataset)...


Horizons:  30%|███       | 3/10 [00:24<00:58,  8.37s/it]

In [ ]:
print("QRF Base Data averages")

display(qrf_base_results.mean())

QRF Base Data averages


NameError: name 'qrf_base_results' is not defined